In [18]:
from typing import TypedDict, Literal
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph

In [2]:
class FeedbackState(TypedDict):
    feedback:str
    sentiment_class:str
    answer:str
    history : list[dict]

In [4]:
llm = ChatOllama(model="llama3.2:3b", teperature=0)

In [6]:
def classifier_agent(state:FeedbackState)->FeedbackState:
    """
    Classifier node to classify the incoming feedback as either positive, negative or neutral
    """
    prompt = f"""
    Classify the customer feedback as one of three categories:
    - Positive
    - Neutral
    - Negative

    Feedback: {state["feedback"]}

    Respond only with one category name. Don't add any punctuations or explanations
    """

    result = llm.invoke(prompt).content.strip()

    state["sentiment_class"] = result

    if not state.get("history"):
        state["history"] = []

    state["history"].append({
        "role":"classifier",
        "content":f"Classified as {result}"
    })

    return state

In [9]:
classifier_agent({"feedback":"This is okay"})

{'feedback': 'This is okay',
 'sentiment_class': 'Neutral',
 'history': [{'role': 'classifier', 'content': 'Classified as Neutral'}]}

In [11]:
def positive_node(state:FeedbackState)->FeedbackState:
    answer = "Thank you for the positive feedback"
    state["answer"] = answer
    state["history"].append({
        "role":"positive_node",
        "content":answer
    })
    return state

def neutral_node(state:FeedbackState)->FeedbackState:
    answer = "Thank you for sharing your experience. We will strive to improve"
    state["answer"] = answer
    state["history"].append({
        "role":"neutral_node",
        "content":answer
    })
    return state

def negative_node(state:FeedbackState)->FeedbackState:
    answer = "Sorry for your inconvenience. We will get back to you soon"
    state["answer"] = answer
    state["history"].append({
        "role":"negative_node",
        "content":answer
    })
    return state

In [15]:
def routing_logic(state:FeedbackState)->Literal["positive", "neutral", "negative"]:
    if state["sentiment_class"].lower() == "positive":
        return "positive"
    elif  state["sentiment_class"].lower() == "neutral":
        return "neutral"
    else:
        return "negative"

In [21]:
builder = StateGraph(FeedbackState)

builder.add_node("classifier", classifier_agent)
builder.add_node("positive", positive_node)
builder.add_node("neutral", neutral_node)
builder.add_node("negative", negative_node)

builder.set_entry_point("classifier")

builder.add_conditional_edges("classifier", routing_logic)

graph = builder.compile()
print(graph.get_graph().draw_ascii())


                      +-----------+                        
                      | __start__ |                        
                      +-----------+                        
                            *                              
                            *                              
                            *                              
                      +------------+                       
                      | classifier |                       
                    ..+------------+..                     
                ....        .         ....                 
            ....            .             ....             
          ..                .                 ..           
+----------+           +---------+           +----------+  
| negative |           | neutral |           | positive |  
+----------+****       +---------+        ***+----------+  
                ****        *         ****                 
                    ****    *     ****  

In [23]:
graph.invoke({"feedback":"This is amazing"})["answer"]

'Thank you for the positive feedback'